In [1]:
# For tips on running notebooks in Google Colab, see
# https://pytorch.org/tutorials/beginner/colab
%matplotlib inline

In [2]:
# prompt: actualizar pytorch

!pip install --upgrade torch torchvision torchaudio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| **Build Model** \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Build the Neural Network
========================

Neural networks comprise of layers/modules that perform operations on
data. The [torch.nn](https://pytorch.org/docs/stable/nn.html) namespace
provides all the building blocks you need to build your own neural
network. Every module in PyTorch subclasses the
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
A neural network is a module itself that consists of other modules
(layers). This nested structure allows for building and managing complex
architectures easily.

In the following sections, we\'ll build a neural network to classify
images in the FashionMNIST dataset.


In [3]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

Get Device for Training
=======================

We want to be able to train our model on an
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [ ]:
device ="gpu:0"

Define the Class
================

We define our neural network by subclassing `nn.Module`, and initialize
the neural network layers in `__init__`. Every `nn.Module` subclass
implements the operations on input data in the `forward` method.


In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

We create an instance of `NeuralNetwork`, and move it to the `device`,
and print its structure.


In [6]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model\'s
`forward`, along with some [background
operations](https://github.com/pytorch/pytorch/blob/270111b7b611d174967ed204776985cefca9c144/torch/nn/modules/module.py#L866).
Do not call `model.forward()` directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0
corresponding to each output of 10 raw predicted values for each class,
and dim=1 corresponding to the individual values of each output. We get
the prediction probabilities by passing it through an instance of the
`nn.Softmax` module.


In [7]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([4], device='cuda:0')


------------------------------------------------------------------------


Model Layers
============

Let\'s break down the layers in the FashionMNIST model. To illustrate
it, we will take a sample minibatch of 3 images of size 28x28 and see
what happens to it as we pass it through the network.


In [8]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


nn.Flatten
==========

We initialize the
[nn.Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html)
layer to convert each 2D 28x28 image into a contiguous array of 784
pixel values ( the minibatch dimension (at dim=0) is maintained).


In [9]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


nn.Linear
=========

The [linear
layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
is a module that applies a linear transformation on the input using its
stored weights and biases.


In [10]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


nn.ReLU
=======

Non-linear activations are what create the complex mappings between the
model\'s inputs and outputs. They are applied after linear
transformations to introduce *nonlinearity*, helping neural networks
learn a wide variety of phenomena.

In this model, we use
[nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
between our linear layers, but there\'s other activations to introduce
non-linearity in your model.


In [11]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.6086,  0.4546, -0.2715, -0.4184, -0.1733,  0.0469, -0.1004, -0.2384,
          0.4787, -0.1910, -0.2357,  0.0936,  0.3072,  0.1971,  0.2497, -0.2552,
          0.5155, -0.6182, -0.2296, -0.0224],
        [-0.6345,  0.1462, -0.4670, -0.2541,  0.0260, -0.0221,  0.1845, -0.3246,
          0.4080,  0.0431,  0.2669, -0.1144,  0.4366, -0.1418,  0.2871, -0.1506,
          0.2317, -0.4755, -0.0331,  0.0364],
        [-0.3789,  0.3096, -0.6804, -0.5915,  0.1626,  0.4010,  0.2812, -0.4087,
          0.3396, -0.2525, -0.0296,  0.3066,  0.5986,  0.2884,  0.1709, -0.1591,
          0.1205, -0.6016, -0.1760,  0.0240]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.4546, 0.0000, 0.0000, 0.0000, 0.0469, 0.0000, 0.0000, 0.4787,
         0.0000, 0.0000, 0.0936, 0.3072, 0.1971, 0.2497, 0.0000, 0.5155, 0.0000,
         0.0000, 0.0000],
        [0.0000, 0.1462, 0.0000, 0.0000, 0.0260, 0.0000, 0.1845, 0.0000, 0.4080,
         0.0431, 0.2669, 0.0000, 0.4366, 0.0000, 0.28

nn.Sequential
=============

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
is an ordered container of modules. The data is passed through all the
modules in the same order as defined. You can use sequential containers
to put together a quick network like `seq_modules`.


In [13]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)
logits

tensor([[-0.2309, -0.1353,  0.0344, -0.0737, -0.3334, -0.1770, -0.1821,  0.3006,
          0.0714, -0.0391],
        [-0.2419, -0.1600, -0.0578, -0.0840, -0.2647, -0.0947, -0.1372,  0.2519,
          0.0408, -0.0350],
        [-0.2237, -0.1557,  0.0135, -0.0723, -0.3080, -0.0839, -0.0989,  0.2994,
          0.0358, -0.1140]], grad_fn=<AddmmBackward0>)

nn.Softmax
==========

The last linear layer of the neural network returns [logits]{.title-ref}
- raw values in \[-infty, infty\] - which are passed to the
[nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
module. The logits are scaled to values \[0, 1\] representing the
model\'s predicted probabilities for each class. `dim` parameter
indicates the dimension along which the values must sum to 1.


In [14]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)
pred_probab

tensor([[0.0844, 0.0929, 0.1100, 0.0988, 0.0762, 0.0891, 0.0886, 0.1436, 0.1142,
         0.1023],
        [0.0840, 0.0912, 0.1010, 0.0984, 0.0821, 0.0974, 0.0933, 0.1377, 0.1115,
         0.1033],
        [0.0847, 0.0907, 0.1074, 0.0986, 0.0779, 0.0974, 0.0960, 0.1429, 0.1098,
         0.0945]], grad_fn=<SoftmaxBackward0>)

Model Parameters
================

Many layers inside a neural network are *parameterized*, i.e. have
associated weights and biases that are optimized during training.
Subclassing `nn.Module` automatically tracks all fields defined inside
your model object, and makes all parameters accessible using your
model\'s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and
a preview of its values.


In [15]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0289,  0.0169,  0.0311,  ...,  0.0089, -0.0290, -0.0301],
        [-0.0293, -0.0178, -0.0127,  ..., -0.0228, -0.0307,  0.0174]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0218, -0.0048], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0116, -0.0354,  0.0286,  ..., -0.0161, -0.0177,  0.0338],
        [ 0.0172,  0.0046,  0.0038,  ..., -0.0307,  0.0332, -0.0287]],
       device='cuda:0', grad_fn=<Sl

------------------------------------------------------------------------


Further Reading
===============

-   [torch.nn API](https://pytorch.org/docs/stable/nn.html)


In [22]:
# prompt: actualiza keras

!pip install --upgrade tensorflow
!pip install --upgrade keras


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.4 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 3.8.0
    Uninstalling keras-3.8.0:
      Successfully uninstalled keras-3.8.0


In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
# Import necessary modules
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from math import sqrt
# Keras specific
import tensorflow as tf  # Import TensorFlow
from tensorflow import keras  # Import Keras from TensorFlow
from keras.models import Sequential
from keras.layers import Dense


# ... (rest of your code remains unchanged) ...

In [3]:
#Ponemos nombre a las columnas y cargamos el dataset
column_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
df = pd.read_csv("housing.csv",  delimiter=r"\s+", header=None, names=column_names)
print(df.shape)
df.describe()
#Ponemos com target la columna MEDV
target_column = ['MEDV']
#Separamos el target de los predictores
predictors = list(set(list(df.columns))-set(target_column))
df[predictors] = df[predictors]/df[predictors].max()
df.describe()

(506, 14)


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
count,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000
mean,0.040612,0.113636,0.401470,0.069170,0.636849,0.715790,0.685749,0.312954,0.397892,0.574173,0.838888,0.898650,0.333238,22.532806
std,0.096672,0.233225,0.247309,0.253994,0.133040,0.080025,0.281489,0.173645,0.362802,0.237042,0.098407,0.230020,0.188071,9.197104
min,0.000071,0.000000,0.016583,0.000000,0.442021,0.405581,0.029000,0.093151,0.041667,0.263010,0.572727,0.000806,0.045562,5.000000
25%,0.000922,0.000000,0.187094,0.000000,0.515499,0.670330,0.450250,0.173189,0.166667,0.392405,0.790909,0.945773,0.183039,17.025000
50%,0.002883,0.000000,0.349315,0.000000,0.617681,0.707118,0.775000,0.264499,0.208333,0.464135,0.865909,0.986243,0.299184,21.200000
75%,0.041327,0.125000,0.652487,0.000000,0.716418,0.754385,0.940750,0.427858,1.000000,0.936709,0.918182,0.998299,0.446537,25.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,50.000000


In [4]:
#Separamos el dataset en train y test
X = df[predictors].values
y = df[target_column].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=40)
print(X_train.shape); print(X_test.shape)

(354, 13)
(152, 13)


In [5]:
# Definimos el modelo
model = Sequential()
model.add(Dense(500, input_dim=13, activation= "relu"))
model.add(Dense(100, activation= "relu"))
model.add(Dense(50, activation= "relu"))
model.add(Dense(1))
model.summary() #Print model Summary

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 500)                 │           7,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 100)                 │          50,100 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 50)                  │           5,050 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              51 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 62,201 (242.97 KB)

 Trainable params: 62,201 (242.97 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# prompt: ya tenemos el modelo definido en la celda anterior, ahora lo preparamos, compilamos añadiendo función de pérdida, optimizador etc., entrenamos, ,probamos, clasificamos y obtenemos el error

# Compile the model
model.compile(loss='mean_squared_error', optimizer='adam') #,metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=200)

# Predict on the test set
predictions = model.predict(X_test)

# Evaluate the model
rmse = sqrt(mean_squared_error(y_test, predictions))
print('RMSE:', rmse)

Epoch 1/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 64ms/step - loss: 569.8820
Epoch 2/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 533.0717  
Epoch 3/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 317.3962 
Epoch 4/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 109.4835
Epoch 5/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 94.4723 
Epoch 6/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 71.5052 
Epoch 7/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 69.2938  
Epoch 8/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 66.9135 
Epoch 9/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 51.2383 
Epoch 10/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 56.2069
Epoch 11/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 49.2883 
Epoch 12/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 45.4712 
Epoch 13/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 37.5195 
Epoch 14/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 37.5523  
Epoch 15/200
12/12 ━━━━━━━━━━━━━━━━━━

In [9]:
for i in range(10):
  print(predictions[i], y_test[i])

[25.008364] [22.7]
[27.191034] [30.3]
[17.166388] [14.4]
[15.248096] [13.4]
[17.836836] [20.1]
[44.2748] [50.]
[24.531387] [24.7]
[17.086927] [17.8]
[18.401506] [17.]
[6.9955144] [7.]


Pytorch

In [10]:
# Import required libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from math import sqrt

In [11]:
# Ponemos nombre a las columnas y cargamos el dataset
column_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
df = pd.read_csv("housing.csv",  delimiter=r"\s+", header=None, names=column_names)
print(df.shape)
df.describe()
# Ponemos com target la columna MEDV
target_column = ['MEDV']
# Separamos el target de los predictores
predictors = list(set(list(df.columns))-set(target_column))
df[predictors] = df[predictors]/df[predictors].max()
df.describe()

(506, 14)


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
count,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000
mean,0.040612,0.113636,0.401470,0.069170,0.636849,0.715790,0.685749,0.312954,0.397892,0.574173,0.838888,0.898650,0.333238,22.532806
std,0.096672,0.233225,0.247309,0.253994,0.133040,0.080025,0.281489,0.173645,0.362802,0.237042,0.098407,0.230020,0.188071,9.197104
min,0.000071,0.000000,0.016583,0.000000,0.442021,0.405581,0.029000,0.093151,0.041667,0.263010,0.572727,0.000806,0.045562,5.000000
25%,0.000922,0.000000,0.187094,0.000000,0.515499,0.670330,0.450250,0.173189,0.166667,0.392405,0.790909,0.945773,0.183039,17.025000
50%,0.002883,0.000000,0.349315,0.000000,0.617681,0.707118,0.775000,0.264499,0.208333,0.464135,0.865909,0.986243,0.299184,21.200000
75%,0.041327,0.125000,0.652487,0.000000,0.716418,0.754385,0.940750,0.427858,1.000000,0.936709,0.918182,0.998299,0.446537,25.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,50.000000


In [12]:
# Separamos el dataset en train y test
X = df[predictors].values
y = df[target_column].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=40)
print(X_train.shape); print(X_test.shape)

# Convertimos los datos a tensores
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

(354, 13)
(152, 13)


In [14]:
# Definimos el modelo
class NeuralNet(nn.Module):
    def __init__(self):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(13, 500)
        self.fc2 = nn.Linear(500, 100)
        self.fc3 = nn.Linear(100, 50)
        self.fc4 = nn.Linear(50, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.relu(out)
        out = self.fc4(out)
        return out

model = NeuralNet()

In [15]:
# Definimos la función de pérdida y el optimizador
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [18]:
# Entrenamos el modelo
num_epochs = 350
for epoch in range(num_epochs):
    model.train()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [20/350], Loss: 11.8182
Epoch [40/350], Loss: 11.6007
Epoch [60/350], Loss: 11.3897
Epoch [80/350], Loss: 11.1795
Epoch [100/350], Loss: 10.9683
Epoch [120/350], Loss: 10.7613
Epoch [140/350], Loss: 10.5608
Epoch [160/350], Loss: 10.3538
Epoch [180/350], Loss: 10.1469
Epoch [200/350], Loss: 9.9295
Epoch [220/350], Loss: 9.6958
Epoch [240/350], Loss: 9.4818
Epoch [260/350], Loss: 9.2750
Epoch [280/350], Loss: 9.0670
Epoch [300/350], Loss: 8.8292
Epoch [320/350], Loss: 8.6152
Epoch [340/350], Loss: 8.4061


In [19]:
# Predecimos en el conjunto de prueba
model.eval()
with torch.no_grad():
    predictions = model(X_test)

In [20]:
# Evaluamos el modelo
rmse = sqrt(mean_squared_error(y_test.numpy(), predictions.numpy()))
print('RMSE:', rmse)

RMSE: 3.6876148270654525


In [21]:
# prompt: imprime las primeras 10 predicciones con el resultado real al lado

for i in range(10):
  print(f"Prediction: {predictions[i].item():.4f}, Actual: {y_test[i].item():.4f}")


Prediction: 21.3785, Actual: 22.7000
Prediction: 29.1946, Actual: 30.3000
Prediction: 16.6070, Actual: 14.4000
Prediction: 14.8391, Actual: 13.4000
Prediction: 16.9913, Actual: 20.1000
Prediction: 45.1105, Actual: 50.0000
Prediction: 23.2783, Actual: 24.7000
Prediction: 16.7558, Actual: 17.8000
Prediction: 18.0953, Actual: 17.0000
Prediction: 7.0046, Actual: 7.0000
